# 8A Targeted improvement experiments (issue #24)

Builds on `error_analysis.ipynb` (PR #29), whose best model is `ensemble_top2` -- the mean probability of
`modernbert_v2_biasinit` + `modernbert_v1_6ep` -- with per-category best-F1 cutoffs picked on tune
(validation macro F1 0.559). Each experiment here changes **one** thing relative to its parent:

| ID | Change | Parent | Retraining? |
|---|---|---|---|
| E1 | Threshold floor: categories with fewer than N tune positives keep the 0.5 cutoff | `ensemble_top2`, best-F1 cutoffs | no |
| E2 | Learning rate 3e-5 instead of 5e-5 | `modernbert_v2_biasinit` | yes (Colab) |
| E3 | Class weights (`--pos-weight sqrt`) on top of bias init | `modernbert_v2_biasinit` | yes (Colab) |

Same rules as before: every choice is made on **tune**, **validation** is only used to report, `test.json`
is never touched, and all **41** categories are predicted.

**Before running:** `data/processed/` must come from the current `main_pipeline.ipynb` (8549 chunks). Run from
`notebooks/`. Outputs go to `results/improvement_experiments/`.

## Setup

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from eval_metrics import multilabel_metrics
from splits import SPLIT_NAMES, load_splits

REPO_DIR = Path(os.getcwd()).parent
PROCESSED_DIR = REPO_DIR / "data" / "processed"
RESULTS_DIR = REPO_DIR / "results"
OUT_DIR = RESULTS_DIR / "improvement_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_N_CHUNKS = 8549  # same check as modernbert_finetune.py and error_analysis.ipynb

chunks_df = pd.read_parquet(PROCESSED_DIR / "chunks_df.parquet")
chunk_labels_df = pd.read_parquet(PROCESSED_DIR / "chunk_labels_df.parquet")
if len(chunks_df) != EXPECTED_N_CHUNKS:
    raise ValueError(f"{PROCESSED_DIR} has {len(chunks_df)} chunks, expected {EXPECTED_N_CHUNKS} -- stale data "
                     "from the old cl100k chunker, re-run main_pipeline.ipynb first")
chunks_df = chunks_df[chunks_df["chunk_start"] >= 0]

# same chunk table as error_analysis.ipynb (without the text/position columns it only needs for 7B)
label_matrix = chunk_labels_df.pivot_table(
    index="chunk_id", columns="category", values="label", aggfunc="max", fill_value=0
).astype(int)
categories = label_matrix.columns.tolist()
assert len(categories) == 41, "project rules say predict all 41 CUAD categories"

table = chunks_df.set_index("chunk_id")[["contract_id"]].join(label_matrix, how="inner").reset_index()

splits = load_splits()
parts = {name: table[table["contract_id"].isin(set(splits[name]))].reset_index(drop=True) for name in SPLIT_NAMES}
y = {name: parts[name][categories].to_numpy() for name in SPLIT_NAMES}
contract_sets = [set(splits[name]) for name in SPLIT_NAMES]
assert all(not (a & b) for i, a in enumerate(contract_sets) for b in contract_sets[i + 1:]), "a contract is in two splits"
for name in SPLIT_NAMES:
    print(f"{name:>5}: {len(parts[name]):5d} chunks, {parts[name]['contract_id'].nunique():3d} contracts, "
          f"{int(y[name].sum()):5d} positive pairs")

train:  5422 chunks, 260 contracts,  5318 positive pairs
 tune:  1286 chunks,  66 contracts,  1308 positive pairs
  val:  1841 chunks,  82 contracts,  1691 positive pairs


In [2]:
def load_probs(run, split):
    """A run's saved probabilities, re-ordered to line up row for row with parts[split] (from error_analysis.ipynb)."""
    saved = pd.read_parquet(RESULTS_DIR / run / f"{split}_probs.parquet").set_index("chunk_id")
    mismatch = set(parts[split]["chunk_id"]) ^ set(saved.index)
    if mismatch:
        raise ValueError(f"{run}/{split}: {len(mismatch)} chunk ids differ from data/processed -- stale data?")
    return saved.loc[parts[split]["chunk_id"], categories].to_numpy()


# the current best model, pinned by name: error_analysis.ipynb picks its top 2 by tune macro AP over every
# results/modernbert_* folder, so adding the E2/E3 runs could otherwise silently change E1's parent
PARENT_RUNS = ["modernbert_v2_biasinit", "modernbert_v1_6ep"]
probs = {run: {split: load_probs(run, split) for split in ("tune", "val")} for run in PARENT_RUNS}
probs["ensemble_top2"] = {split: np.mean([probs[run][split] for run in PARENT_RUNS], axis=0)
                          for split in ("tune", "val")}
print(f"ensemble_top2 = {' + '.join(PARENT_RUNS)}")

ensemble_top2 = modernbert_v2_biasinit + modernbert_v1_6ep


## Threshold helpers

`pick_thresholds` is `error_analysis.ipynb`'s best-F-beta cutoff picker with one change, the variable E1 tests:
`min_positives`. A category with fewer than `min_positives` positives in the fitting set keeps the 0.5 default
instead of getting a tuned cutoff. `min_positives=1` is exactly the old rule (only categories with 0 positives keep
0.5), so the parent is reproduced first as a check.

In [3]:
DEFAULT_THRESHOLD = 0.5
CANDIDATE_THRESHOLDS = np.round(np.arange(0.01, 0.96, 0.01), 2)  # same grid as error_analysis.ipynb


def pick_thresholds(y_true, scores, beta=1, min_positives=1, n_pos_for_floor=None):
    """One cutoff per category: the candidate with the best F-beta on (y_true, scores), ties to the lowest cutoff.
    Categories with fewer than `min_positives` positives keep the default. `n_pos_for_floor` overrides which counts
    the floor looks at (used in cross-validation, so a fold floors the same categories the full tune set would)."""
    n_pos_all = y_true.sum(axis=0) if n_pos_for_floor is None else np.asarray(n_pos_for_floor)
    picked = np.full(len(categories), DEFAULT_THRESHOLD)
    b2 = beta ** 2
    for i in range(len(categories)):
        n_pos = y_true[:, i].sum()
        if n_pos == 0 or n_pos_all[i] < min_positives:
            continue
        pred = scores[:, i, None] >= CANDIDATE_THRESHOLDS        # (n_chunks x n_candidates)
        tp = (pred & (y_true[:, i, None] == 1)).sum(axis=0)
        fp = pred.sum(axis=0) - tp
        fn = n_pos - tp
        fbeta = (1 + b2) * tp / np.maximum((1 + b2) * tp + b2 * fn + fp, 1)
        picked[i] = CANDIDATE_THRESHOLDS[int(np.argmax(fbeta))]
    return picked


def log_row(experiment_id, parent, model, what, thr):
    """One experiment-log row, same metric columns as results/error_analysis/experiment_log.csv."""
    tune_summary, _ = multilabel_metrics(y["tune"], probs[model]["tune"], categories, thr)
    val_summary, _ = multilabel_metrics(y["val"], probs[model]["val"], categories, thr)
    return {
        "experiment_id": experiment_id,
        "parent": parent,
        "model": model,
        "what": what,
        "tune_macro_ap": tune_summary["macro_ap_present"],
        "tune_macro_f1": tune_summary["macro_f1_present"],
        "val_macro_ap": val_summary["macro_ap_present"],
        "val_micro_precision": val_summary["micro_precision"],
        "val_micro_recall": val_summary["micro_recall"],
        "val_micro_f1": val_summary["micro_f1"],
        "val_macro_f1": val_summary["macro_f1_present"],
    }


# sanity check: min_positives=1 must reproduce error_analysis.ipynb's best model exactly
parent_thr = pick_thresholds(y["tune"], probs["ensemble_top2"]["tune"])
logged = pd.read_csv(RESULTS_DIR / "error_analysis" / "experiment_log.csv")
logged = logged[(logged["model"] == "ensemble_top2") & (logged["thresholds"] == "best F1")].iloc[0]
parent_row = log_row("E0", "", "ensemble_top2", "current best: ensemble_top2, best-F1 cutoffs", parent_thr)
for col in ("val_macro_f1", "val_micro_f1", "val_macro_ap"):
    assert np.isclose(parent_row[col], logged[col]), f"{col}: {parent_row[col]:.4f} vs logged {logged[col]:.4f}"
print(f"parent reproduced: val macro F1 {parent_row['val_macro_f1']:.3f}, micro F1 {parent_row['val_micro_f1']:.3f}, "
      f"macro AP {parent_row['val_macro_ap']:.3f}")

parent reproduced: val macro F1 0.559, micro F1 0.664, macro AP 0.593


## E1 Threshold floor

**Why:** 7B found that most false alarms come from one category. Affiliate License-Licensor has only 2 positives
in tune, so its best-F1 cutoff landed at 0.01. That makes it fire on License Grant / Non-Transferable License chunks
(~170 FPs in validation). A cutoff fit on a handful of examples is mostly noise. The fix is to tune a cutoff only
when there's enough evidence, and otherwise fall back to 0.5.

**How N is chosen -- cross-validation on tune.** Picking N by plain tune F1 would always choose "no floor", because
cutoffs fit on tune always score best on tune. Instead, the 66 tune contracts are split into 5 folds (by contract).
For each fold, cutoffs are fit on the other 4 and applied to the held-out fold. The held-out predictions from all 5
folds are pooled into one out-of-fold prediction for every tune chunk, and N is chosen by macro F1 (present
categories) on that. The floor counts positives on the **full** tune set, so each fold floors exactly the
categories the final rule will. Ties go to the smallest N (the least change from the parent). Validation is not
looked at until N is fixed.

In [4]:
FLOOR_GRID = [1, 3, 5, 10, 15, 20]  # 1 = parent rule (only categories with 0 positives keep 0.5)
N_FOLDS = 5

tune_scores = probs["ensemble_top2"]["tune"]
tune_n_pos = y["tune"].sum(axis=0)
folds = list(GroupKFold(n_splits=N_FOLDS).split(tune_scores, groups=parts["tune"]["contract_id"]))

sweep = []
for floor in FLOOR_GRID:
    oof_pred = np.zeros_like(y["tune"])
    for fit_idx, held_idx in folds:
        thr = pick_thresholds(y["tune"][fit_idx], tune_scores[fit_idx], min_positives=floor, n_pos_for_floor=tune_n_pos)
        oof_pred[held_idx] = tune_scores[held_idx] >= thr
    # oof_pred is already 0/1, so scoring it at 0.5 just reads it back
    cv_summary, _ = multilabel_metrics(y["tune"], oof_pred, categories)
    sweep.append({
        "min_positives": floor,
        "n_categories_floored": int(((tune_n_pos > 0) & (tune_n_pos < floor)).sum()),
        "cv_tune_macro_f1": cv_summary["macro_f1_present"],
        "cv_tune_micro_f1": cv_summary["micro_f1"],
    })
sweep_df = pd.DataFrame(sweep)
best_floor = int(sweep_df.loc[sweep_df["cv_tune_macro_f1"].idxmax(), "min_positives"])  # idxmax: first = smallest N
print(f"chosen on tune (cross-validated): min_positives = {best_floor}\n")
print(sweep_df.round(3).to_string(index=False))

chosen on tune (cross-validated): min_positives = 1

 min_positives  n_categories_floored  cv_tune_macro_f1  cv_tune_micro_f1
             1                     0             0.540             0.657
             3                     2             0.538             0.688
             5                     5             0.529             0.695
            10                     9             0.530             0.703
            15                    14             0.503             0.706
            20                    15             0.495             0.702


### E1 on validation (report only)

Final cutoffs are fit on the full tune set with the chosen floor, then scored once on validation against the parent.
The whole sweep is also shown on validation for transparency, but it played no part in choosing N.

In [5]:
e1_thr = pick_thresholds(y["tune"], tune_scores, min_positives=best_floor)
e1_row = log_row("E1", "E0", "ensemble_top2",
                 f"threshold floor: categories with < {best_floor} tune positives keep 0.5 (N picked by 5-fold CV on tune)",
                 e1_thr)
compare = pd.DataFrame([parent_row, e1_row]).set_index("experiment_id")
print(compare.drop(columns=["parent", "model", "what"]).T.round(3).to_string(), "\n")

# report-only: every floor on validation
val_sweep = []
for floor in FLOOR_GRID:
    s, _ = multilabel_metrics(y["val"], probs["ensemble_top2"]["val"], categories,
                              pick_thresholds(y["tune"], tune_scores, min_positives=floor))
    val_sweep.append({"min_positives": floor, "val_macro_f1": s["macro_f1_present"], "val_micro_f1": s["micro_f1"],
                      "val_micro_precision": s["micro_precision"], "val_micro_recall": s["micro_recall"]})
sweep_df = sweep_df.merge(pd.DataFrame(val_sweep), on="min_positives")
sweep_df.to_csv(OUT_DIR / "e1_floor_sweep.csv", index=False)
print("full sweep (val columns are report-only):")
print(sweep_df.round(3).to_string(index=False))

experiment_id           E0     E1
tune_macro_ap        0.590  0.590
tune_macro_f1        0.604  0.604
val_macro_ap         0.593  0.593
val_micro_precision  0.633  0.633
val_micro_recall     0.698  0.698
val_micro_f1         0.664  0.664
val_macro_f1         0.559  0.559 



full sweep (val columns are report-only):
 min_positives  n_categories_floored  cv_tune_macro_f1  cv_tune_micro_f1  val_macro_f1  val_micro_f1  val_micro_precision  val_micro_recall
             1                     0             0.540             0.657         0.559         0.664                0.633             0.698
             3                     2             0.538             0.688         0.559         0.696                0.699             0.694
             5                     5             0.529             0.695         0.545         0.700                0.710             0.691
            10                     9             0.530             0.703         0.545         0.703                0.720             0.687
            15                    14             0.503             0.706         0.522         0.712                0.761             0.669
            20                    15             0.495             0.702         0.526         0.715                0.

**Result:** cross-validated tune macro F1 is highest with no floor (N = 1), so E1 as selected is identical to
its parent -- on the metric chosen up front, the floor **didn't help**.

**Noted alternative, N = 3 (seen after validation, not selected):** it floors only the 2 categories with 2 tune
positives, which keep 0.5 instead of a tuned cutoff:

- **Affiliate License-Licensor** (2 tune / 7 val positives, tuned cutoff 0.01) -- 137 validation FPs drop to 5, but recall falls from 7/7 to 1/7
- **Most Favored Nation** (2 tune / 7 val positives, tuned cutoff 0.02) -- 46 validation FPs drop to 0, recall 1/7 -> 0/7

Together that's 684 -> 506 validation false positives (-178).

Macro F1 is flat (CV tune 0.538 vs 0.540; validation 0.559 both), while micro F1 and precision rise
(validation micro F1 0.664 -> 0.696, precision 0.633 -> 0.699). It's a precision option for the team to weigh, not
a tune-selected result. The table below shows what it changes per category (report-only).

In [6]:
# per category, report-only: what the noted N = 3 floor changes vs. the parent
ALT_FLOOR = 3
alt_thr = pick_thresholds(y["tune"], tune_scores, min_positives=ALT_FLOOR)
val_scores = probs["ensemble_top2"]["val"]
y_val = y["val"]
changed = np.flatnonzero(~np.isclose(parent_thr, alt_thr))
_, parent_cat = multilabel_metrics(y_val, val_scores, categories, parent_thr)
_, alt_cat = multilabel_metrics(y_val, val_scores, categories, alt_thr)


def fp_count(thr):
    return ((val_scores >= thr) & (y_val == 0)).sum(axis=0)


floored_df = pd.DataFrame({
    "category": np.array(categories)[changed],
    "n_pos_tune": tune_n_pos[changed],
    "n_pos_val": y_val.sum(axis=0)[changed],
    "cutoff_parent": parent_thr[changed],
    f"cutoff_n{ALT_FLOOR}": alt_thr[changed],
    "fp_parent": fp_count(parent_thr)[changed],
    f"fp_n{ALT_FLOOR}": fp_count(alt_thr)[changed],
    "recall_parent": parent_cat["recall"].to_numpy()[changed],
    f"recall_n{ALT_FLOOR}": alt_cat["recall"].to_numpy()[changed],
    "f1_parent": parent_cat["f1"].to_numpy()[changed],
    f"f1_n{ALT_FLOOR}": alt_cat["f1"].to_numpy()[changed],
}).sort_values("n_pos_tune")
floored_df.to_csv(OUT_DIR / f"e1_floored_categories_n{ALT_FLOOR}.csv", index=False)
pd.DataFrame({"category": categories, "threshold_parent": parent_thr, "threshold_e1": e1_thr,
              f"threshold_n{ALT_FLOOR}": alt_thr}).to_csv(OUT_DIR / "e1_selected_thresholds.csv", index=False)

total_fp_parent, total_fp_alt = int(fp_count(parent_thr).sum()), int(fp_count(alt_thr).sum())
print(f"N = {ALT_FLOOR} (report-only): validation false positives {total_fp_parent} -> {total_fp_alt} "
      f"({total_fp_alt - total_fp_parent:+d}), from flooring {len(changed)} categories: "
      f"{', '.join(floored_df['category'])}\n")
print(floored_df.round(2).to_string(index=False))

N = 3 (report-only): validation false positives 684 -> 506 (-178), from flooring 2 categories: Affiliate License-Licensor, Most Favored Nation

                  category  n_pos_tune  n_pos_val  cutoff_parent  cutoff_n3  fp_parent  fp_n3  recall_parent  recall_n3  f1_parent  f1_n3
Affiliate License-Licensor           2          7           0.01        0.5        137      5           1.00       0.14       0.09   0.15
       Most Favored Nation           2          7           0.02        0.5         46      0           0.14       0.00       0.04   0.00


In [7]:
# experiment log for #24 -- E2/E3 rows are added once their Colab runs are in results/
experiment_log = pd.DataFrame([parent_row, e1_row])
experiment_log.to_csv(OUT_DIR / "experiment_log.csv", index=False)
print(experiment_log.drop(columns="what").round(3).to_string(index=False))

experiment_id parent         model  tune_macro_ap  tune_macro_f1  val_macro_ap  val_micro_precision  val_micro_recall  val_micro_f1  val_macro_f1
           E0        ensemble_top2           0.59          0.604         0.593                0.633             0.698         0.664         0.559
           E1     E0 ensemble_top2           0.59          0.604         0.593                0.633             0.698         0.664         0.559
